In [1]:
import wave 
import pyroomacoustics as pa
import numpy as np
import scipy.signal as sp
import scipy

from calc_steering_vector import calculate_steering_vector

In [2]:
# 遅延和アレイ
def execute_two_microphone_sparse_separation(x_left, x_right, is_amplitude_enabled=False):
    """
    x_left: 左の音源に近いマイクロホン (freq_bins, time_frames)
    x_right: 右の音源に近いマイクロホン (freq_bins, time_frames)
    is_amplitude_enabled: 振幅比を用いて分離を行う場合はTrue
    """
    if is_amplitude_enabled == True:
        # 振幅比を用いた分離
        amp_ratio = np.abs(x_left) / np.maximum(np.abs(x_right), 1.e-18)
        y_left = (amp_ratio > 1.).astype(np.float) * x_left
        y_right = (amp_ratio < 1.).astype(np.float) * x_right
    else:
        # 位相差を用いた分離
        phase_difference = np.angle(x_left/x_right)
        y_left = (phase_difference > 0.).astype(np.float) * x_left
        y_right = (phase_difference < 0.).astype(np.float) * x_right
    """y_left: (freq_bins, time_frames), y_right: (freq_bins, time_frames)"""
    return (y_left, y_right)

In [3]:
# SNRを測る
def calculate_snr(target, out):
    """
    target: 目的音 (num_samples, )
    out: 雑音除去後の信号 (num_samples, )
    """
    wave_length = np.minimum(np.shape(target)[0], np.shape(out)[0])
    # 消し残った雑音
    target = target[:wave_length]
    out = out[:wave_length]
    noise = target - out
    snr = 10. * np.log10(np.sum(np.square(target)) / np.sum(np.square(noise)))
    return snr

In [8]:
if __name__ == "__main__":
    # 乱数の種を初期化
    np.random.seed(0)
    # 畳み込みに用いる波形
    clean_wave_files = ["./CMU_ARCTIC/cmu_us_aew/wav/arctic_a0001.wav", "./CMU_ARCTIC/cmu_us_axb/wav/arctic_a0002.wav"]
    # 雑音だけの区間のフレーム数
    n_noise_only = 40000
    # 音源数
    n_sources = len(clean_wave_files)
    # 音声波形の長さを調べる
    n_samples = 0
    # ファイルを読み込む
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        if n_samples<wav.getnframes():
            n_samples=wav.getnframes()
        wav.close()
    clean_data = np.zeros([n_sources, n_samples])

    # ファイルを読み込む
    s = 0
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        data = wav.readframes(wav.getnframes())
        data = np.frombuffer(data, dtype=np.int16)
        data = data/np.iinfo(np.int16).max
        clean_data[s, :wav.getnframes()] = data
        wav.close()
        s = s+1

    # シミュレーションのパラメータ
    n_sim_sources = 2
    # サンプリングレート [Hz]
    sample_rate = 16000
    # フレームサイズ
    N = 1024
    # 周波数の数
    Nk = N / 2 + 1
    # 各ビンの周波数
    freqs = np.arange(0, Nk, 1) * sample_rate / N
    # 音声と雑音の比率 [dB]
    SNR = 10.
    # 部屋の大きさ
    room_dim = np.r_[10.0, 10.0, 10.0]
    # マイクロホンアレイを置く部屋の場所
    mic_array_loc = room_dim / 2 + np.random.randn(3) * 0.1
    # # マイクロホンアレイのマイクロホン配置（Far-field）
    # mic_alignments = np.array(
    #         [[x, 0.0, 0.0] for x in np.arange(-0.01, 0.02, 0.02)]
    # )
    # マイクロホンアレイのマイクロホン配置（Near-field）
    mic_alignments = np.array(
            [[x, 0.0, 0.0] for x in np.arange(-0.2, 0.21, 0.4)]
    )
    # マイクロホン数
    n_channels = np.shape(mic_alignments)[0]
    # get the microphone array
    R  = mic_alignments.T + mic_array_loc[:, None]
    """R: (3D-coordinate(x,y,z)=3, num_microphones)"""
    # 部屋を生成する
    room = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    room_no_noise_left = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    room_no_noise_right = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    # 用いるマイクロホンアレイの情報を設置する
    room.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_left.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_right.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    # 音源の場所
    doas  = np.array(
        [[np.pi/2, np.pi],
        [np.pi/2, 0]]
        )
    # 音源とマイクロホンの距離
    distance = 1.
    source_locations = np.zeros((3, doas.shape[0]), dtype=doas.dtype)
    """source_locations: (xyz, num_sources)"""
    source_locations[0,  :] = np.cos(doas[:, 1]) * np.sin(doas[:, 0]) 
    source_locations[1,  :] = np.sin(doas[:, 1]) * np.sin(doas[:, 0])
    source_locations[2,  :] = np.cos(doas[:, 0])
    source_locations *= distance
    source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置
    # print("clean_data:", np.shape(clean_data))

    # 各音源をシミュレーションに追加する
    for s in range(n_sim_sources):
        clean_data[s] /= np.std(clean_data[s])
        room.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 0:
            room_no_noise_left.add_source(source_locations[:, s], signal=clean_data[s])
        elif s == 1:
            room_no_noise_right.add_source(source_locations[:, s], signal=clean_data[s])

    # シミュレーションを回す
    room.simulate(snr=SNR)
    room_no_noise_left.simulate(snr=90)
    room_no_noise_right.simulate(snr=90)

    # 畳み込んだ波形を取得する
    multi_conv_data = room.mic_array.signals
    """multi_conv_data: (num_channels, num_samples)"""
    multi_conv_data_left_no_noise = room_no_noise_left.mic_array.signals
    """multi_conv_data_left_no_noise: (num_channels, num_samples)"""
    multi_conv_data_right_no_noise = room_no_noise_right.mic_array.signals
    """multi_conv_data_right_no_noise: (num_channels, num_samples)"""

    # 短時間フーリエ変換
    f, t, stft_data = sp.stft(multi_conv_data, fs=sample_rate, window="hann", nperseg=N)
    """f: (freq_bins,), t: (1,), stft_data:(num_microphones, freq_bins, time_frames)"""

    # 位相差もしくは振幅比で分離
    y_phase_left, y_phase_right = execute_two_microphone_sparse_separation(stft_data[0, ...], stft_data[1, ...], False)
    y_amp_left, y_amp_right = execute_two_microphone_sparse_separation(stft_data[0, ...], stft_data[1, ...], True)

    # 時間領域の波形に戻す
    t, y_phase_left = sp.istft(y_phase_left, fs=sample_rate, window="hann", nperseg=N)
    t, y_phase_right = sp.istft(y_phase_right, fs=sample_rate, window="hann", nperseg=N)
    t, y_amp_left = sp.istft(y_amp_left, fs=sample_rate, window="hann", nperseg=N)
    t, y_amp_right = sp.istft(y_amp_right, fs=sample_rate, window="hann", nperseg=N)

    # SNRを測る
    snr_pre = calculate_snr(multi_conv_data_left_no_noise[0, ...], multi_conv_data[0, ...]) + calculate_snr(multi_conv_data_right_no_noise[1, ...], multi_conv_data[1, ...])
    snr_phase_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], y_phase_left) + calculate_snr(multi_conv_data_right_no_noise[1, ...], y_phase_right)
    snr_amp_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], y_amp_left) + calculate_snr(multi_conv_data_right_no_noise[1, ...], y_amp_right)
    snr_pre /= 2
    snr_phase_post /= 2
    snr_amp_post /= 2

    print("result ΔSNR [dB]")
    print("PHASE:{:.2f}".format(snr_phase_post-snr_pre))
    print("AMPLITUDE:{:.2f}".format(snr_amp_post-snr_pre))

result ΔSNR [dB]
PHASE:-0.54
AMPLITUDE:4.69
